# Re-score v4-mixed-r16 (HF v5) at lambda=0.5

`select_lambda` shipped this adapter at **lambda=0.25**, but not because 0.25
won. The 0.0 -> 0.25 step *lowered* OOD CER (0.022833 -> 0.022612), so that
step's cost/benefit ratio was **negative**, and the elbow test
`ratio > prev_ratio * threshold` compares against a negative bar that every
later positive ratio clears -- the walk broke before it could accept 0.5.
Fixed in `src/pipeline.py`, pinned by
`tests/test_pipeline.py::test_select_lambda_a_free_step_does_not_reject_every_later_lambda`
(re-runs the real sweep rows, now selects 0.5).

On the numbers the sweep already recorded, **lambda=0.5 dominates 0.25**:

| lambda | val CER | OOD CER | in budget (bound 0.0428) |
|---|---|---|---|
| 0.25 (shipped) | 0.05483 | 0.02261 | yes |
| **0.5** | **0.03636** | 0.02619 | yes |
| 0.75 | 0.03316 | 0.03343 | yes, but 11.7x elbow -> correctly rejected |

What the sweep never recorded is **English-token retention**, the axis
production actually regressed on (`team` -> `tim`). Retention rises with
adapter strength everywhere it has been measured -- base (lambda=0) 0.5574
synthetic / 0.3638 youtube / 0.3158 real, versus v4-mixed-r16 at lambda=0.25
scoring 0.7599 / 0.7094 / 0.4545 -- so 0.5 is *expected* to retain more. This
notebook measures it instead of assuming it.

**Inference only. Nothing is trained, nothing is pushed.**

## 1. Clone / update the repo

In [ ]:
import os

REPO_URL = "https://github.com/egoist-minh/Reworkwhisper-finetune.git"
REPO_DIR = "/kaggle/working/Reworkwhisper-finetune"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## 2. Locate attached datasets

Attach three Kaggle Datasets before running:

1. **`lambda-sweep-artifacts`** -- upload `Outputs/lambda-sweep-artifacts.zip`
   from the repo (107 MB): v4-mixed-r16's `config.json`, `validated_manifest.jsonl`,
   `audit/predictions_baseline_test.csv`, and `checkpoints/best/` (the raw
   pre-lambda-bake adapter -- `adapter/` is already baked to 0.25 and cannot be
   rescaled *up*).
2. **`youtube-meetings`** and **`paid-dataset-v2`** -- the two corpora
   `mixed-noisy-v1` is merged from. The test split spans both sources
   (228 youtube + 426 synthetic = 654 segments), so both audio trees are needed.
3. **`real-meetings-bench`** -- optional, only for the tier-4a number.

Mount nesting is inconsistent between datasets; read the paths off the cell
below rather than assuming `<slug>/<slug>`.

In [ ]:
!ls -la /kaggle/input
!ls -la /kaggle/input/datasets/*/*

## 3. Set platform-specific paths

Edit these to match what the cell above printed. This is the only place a
`/kaggle/input/...` path is written -- `scripts/eval_v4_mixed_at_lambda.py`
never hardcodes one.

In [ ]:
ARTIFACTS   = "/kaggle/input/datasets/<user>/lambda-sweep-artifacts/lambda-sweep-artifacts"  # edit
YOUTUBE_SRC = "/kaggle/input/datasets/<user>/youtube-meetings/youtube-meetings"              # edit
PAID_SRC    = "/kaggle/input/datasets/<user>/paid-dataset-v2/paid-dataset-v2"                # edit
REAL_BENCH  = "/kaggle/input/datasets/<user>/real-meetings-bench/real-meetings-bench"        # edit, or None

RUN_DIR = f"{ARTIFACTS}/v4-mixed-r16"

for label, path in [("RUN_DIR/config.json", f"{RUN_DIR}/config.json"),
                    ("RUN_DIR/checkpoints/best", f"{RUN_DIR}/checkpoints/best"),
                    ("RUN_DIR/validated_manifest.jsonl", f"{RUN_DIR}/validated_manifest.jsonl"),
                    ("RUN_DIR/audit/predictions_baseline_test.csv",
                     f"{RUN_DIR}/audit/predictions_baseline_test.csv"),
                    ("YOUTUBE_SRC", YOUTUBE_SRC), ("PAID_SRC", PAID_SRC),
                    ("REAL_BENCH", REAL_BENCH)]:
    print(f"{os.path.exists(path) if path else 'skipped'}\t{label}")

## 4. Rebuild `mixed-noisy-v1`

`data.dataset_path` was `/kaggle/working/dataset/mixed-noisy-v1` for this run
and that directory is not persisted anywhere, so the merged audio tree has to
be reassembled. `build_mixed_dataset.py` is the same script the original run
used, and it re-runs every gate (disjoint `meeting_id`, disjoint audio dirs,
`verified: true` on every reviewed record) before copying a byte.

`--out` must not already exist non-empty -- the script refuses to merge twice
on top of itself. Re-running this notebook in a fresh session is fine; re-running
this *cell* in the same session is not.

In [ ]:
MIXED = "/kaggle/working/dataset/mixed-noisy-v1"

!python -m scripts.build_mixed_dataset --sources {PAID_SRC} {YOUTUBE_SRC} --out {MIXED}

AUDIO_ROOT = f"{MIXED}/audio"
print("AUDIO_ROOT exists:", os.path.exists(AUDIO_ROOT))

## 5. Environment

In [ ]:
os.environ["TRANSFORMERS_AUTO_CONVERSION"] = "0"

## 6. Run the eval

Loads `vinai/PhoWhisper-large` + v4-mixed-r16's raw checkpoint, scales to
**lambda=0.5** in memory (`src.lora.set_lambda`, the same call
`stage_sweep_gate` makes for every lambda in a sweep), decodes the 654-segment
test split plus real-meetings-bench, and scores through the gate's own
`_score_by_source` / `score_real` -- so `pass` and `retention_pass` mean
exactly what they would in a real gate run.

The scoring path was verified locally against v4-mixed-r16's stored
predictions and reproduces `gate_results.json` to the digit (youtube
0.0764 / 0.7094, synthetic 0.0258 / 0.7599, real 0.2890 / 0.4545). The decode
path is untested here -- no GPU on the author's machine, and CLAUDE.md's
"Kaggle code never works first try" has a good record.

Drop `--real-bench-path` if that dataset is not attached.

In [ ]:
import subprocess

OUT_DIR = "/kaggle/working/v4-mixed-r16-lambda0.5"

cmd = [
    "python", "-m", "scripts.eval_v4_mixed_at_lambda",
    "--run-dir", RUN_DIR,
    "--audio-root", AUDIO_ROOT,
    "--lam", "0.5",
    "--out-dir", OUT_DIR,
]
if REAL_BENCH:
    cmd += ["--real-bench-path", REAL_BENCH]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise SystemExit(f"eval_v4_mixed_at_lambda failed with exit code {proc.returncode}")

## 7. Read the result

The script prints CER and retention per slice against the shipped lambda=0.25
numbers. Read **both columns**, not CER alone -- shipping on CER alone is the
exact failure this whole investigation traced (SESSIONS.md H4).

- **CER holds or improves AND retention rises on every slice** -> lambda=0.5 is
  a strictly better operating point for the same weights. Re-merge and push it;
  no retraining, no new data. This is the cheap win the fix was chasing.
- **Retention rises but CER regresses somewhere** -> a real trade-off, not a
  free win. Decide per slice which one production cares about; do not average
  them away.
- **Retention does not rise** -> the "retention increases with adapter strength"
  reading is wrong, and lambda is not the lever. Say so plainly and stop --
  `select_lambda` was still buggy and the fix still stands on its own, but it
  buys nothing here.

Whatever the outcome, record the four numbers in `SESSIONS.md` before acting on
them, and keep `Outputs/v4-mixed-r16-lambda0.5/` -- the next comparison will
want the predictions, not just the aggregates.

In [ ]:
!ls -la {OUT_DIR}
!cat {OUT_DIR}/summary.json